# Exercise: consume_03 — aggregation & anomaly detection

**Goal:** build a consumer that doesn't just *print* events but **processes** them:
1. parses the JSON payload
2. maintains running statistics per house and sensor (count, min, max, avg)
3. flags values above a threshold

**Prerequisite:** run `exercise_05_produce_stream.ipynb` (especially Task B — anomaly simulation).

In [ ]:
from confluent_kafka import Consumer
from collections import defaultdict
from datetime import datetime
import json

THRESHOLDS = {
    'strom':  50.0,    # kWh
    'wasser': 200.0,   # Liter
}

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'aggregation-exercise',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['strom', 'wasser'])
print('Consumer ready.')

## Step 1 — read and aggregate (up to 200 events)

**Task:** complete the aggregation inside the loop.
Maintain:
- `totals[topic][house]` — sum of values
- `counts[topic][house]` — number of events
- `anomalies` — list of events with `value > THRESHOLDS[topic]`

In [ ]:
totals    = defaultdict(lambda: defaultdict(float))
counts    = defaultdict(lambda: defaultdict(int))
anomalies = []

messages_read, empty_polls = 0, 0
while messages_read < 200 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    try:
        data  = json.loads(msg.value().decode())
        house = data.get('haus', 'unknown')
        topic = msg.topic()
        value = float(data.get('wert', 0.0))

        # TODO: update totals[topic][house] and counts[topic][house]


        # TODO: if value > THRESHOLDS[topic], append to anomalies:
        #   anomalies.append({'topic': topic, 'house': house, 'value': value, 'offset': msg.offset()})

    except Exception as e:
        print(f'Parse error: {e}')

print(f'Done — processed {messages_read} events.')

## Step 2 — print a summary table

Format:
```
Topic   | House   | Count |    Avg
------------------------------------
strom   | haus_a  |    12 |  7.34 kWh
```

In [ ]:
# TODO: print a summary table
# For each topic and house: avg = totals[topic][house] / counts[topic][house]
# Use 'kWh' for strom and 'L' for wasser


## Step 3 — anomaly report

List all detected anomalies. If none — re-run the anomaly simulator in `exercise_05_produce_stream.ipynb`.

In [ ]:
# TODO: print each anomaly with topic, house, value and percent over the threshold
# Hint: pct_over = (value / THRESHOLDS[topic] - 1) * 100


## Bonus — live aggregation

Build a continuous loop that prints an updated summary every 10 events. Run the streaming producer at the same time and watch the numbers evolve.

> Hint: `from IPython.display import clear_output` to refresh the output in place.

In [ ]:
# TODO: live aggregation loop


In [ ]:
try:
    consumer.close()
except Exception:
    pass
print('Closed.')